In [2]:
from langgraph.graph import StateGraph
from dotenv import load_dotenv
from langchain_ollama import ChatOllama
from typing import TypedDict
from langgraph.constants import END, START
from typing import Annotated
from langgraph.graph.message import add_messages
from langchain_core.messages import BaseMessage, HumanMessage, AIMessage
from langgraph.checkpoint.memory import MemorySaver

In [3]:
class Chat(TypedDict):

    messages: Annotated[list[BaseMessage], add_messages]

In [4]:
model = ChatOllama(
    model = "qwen3:1.7b"
)

In [5]:
def chat_node(state: Chat):

    response = model.invoke(state['messages'])
    return {'messages': [response]}

In [6]:
checkpoint = MemorySaver()
graph = StateGraph(Chat)

graph.add_node("chat_node", chat_node)

graph.add_edge(START, "chat_node")
graph.add_edge("chat_node", END)

chatbot = graph.compile(checkpointer = checkpoint)

In [7]:
# chatbot.invoke({
#     'messages': HumanMessage(content="What is the capital of USA")
# })

In [8]:
thread_id = '1'
while True:

    user_message = input('Type here: ')
    print('User: ', user_message)

    if user_message.strip().lower() in ['exit', 'bye', 'quit']:
        break

    config = {'configurable': {'thread_id': thread_id}}

    response = chatbot.invoke({'messages': [HumanMessage(content=user_message)]}, config=config)
    print('AI: ', response['messages'][-1].content)

User:  hi
AI:  Hello! How can I assist you today? Feel free to ask me anything you need help with! 😊
User:  my name is harshit
AI:  Hello, Harshit! It's great to meet you! 😊 How can I assist you today? Feel free to ask me anything you need help with! 🌟
User:  exit


In [10]:
chatbot.get_state(config=config)
list(chatbot.get_state_history(config=config))

[StateSnapshot(values={'messages': [HumanMessage(content='hi', additional_kwargs={}, response_metadata={}, id='475653aa-cac4-4e5d-ad8a-1af04dabbe44'), AIMessage(content='Hello! How can I assist you today? Feel free to ask me anything you need help with! 😊', additional_kwargs={}, response_metadata={'model': 'qwen3:1.7b', 'created_at': '2026-08-27T09:43:20.281318Z', 'done': True, 'done_reason': 'stop', 'total_duration': 6232809709, 'load_duration': 4625507417, 'prompt_eval_count': 11, 'prompt_eval_duration': 86756000, 'eval_count': 89, 'eval_duration': 1485064000, 'logprobs': None, 'model_name': 'qwen3:1.7b', 'model_provider': 'ollama'}, id='lc_run--01a0429a-275d-7352-b668-4fc13091ec92-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 11, 'output_tokens': 89, 'total_tokens': 100}), HumanMessage(content='my name is harshit', additional_kwargs={}, response_metadata={}, id='1bb3161a-7b6a-4514-a0d7-26cae4b6d7dc'), AIMessage(content="Hello, Harshit! It's great to meet 